# Public portfolio · invsqrt 62Q · DQI 892Q · Chemistry 428Q

The selected portfolio builds on public work. Our DQI update reuses one existing intermediate product, replacing one CCX with two CX gates. The notebook packages pinned files; it does not rerun the circuit verifiers.

| Task / local verification | Qubits | Cost | Reference / cost |
|---|---:|---:|---:|
| Invsqrt · public circuit | 62 | 100,568.49919 | 20.65035× |
| DQI · default seed, 0 / 0 errors | 892 | 379,107,931,472.44 | 64.46571× |
| Chemistry · one local verification | 428 | 326,800,029.87 | 10.26805× |

| Our change | Public parent cost | Candidate cost | Local estimated portfolio gain |
|---|---:|---:|---:|
| DQI · one CCX replaced by two CX | 379,108,516,959.92 | 379,107,931,472.44 | +0.0000123 |

The combined local estimate is **23.91010**, not an observed Kaggle score. The DQI candidate and parent both passed two 102,400-shot runs with zero value and phase errors; the fixed-seed comparison saved one mean Toffoli. Different file-derived samples can change the measured result. Chemistry passed one local run with a narrow error margin. The previous Notebook v11 scored **23.83433** with a different portfolio.


In [ ]:
from pathlib import Path
import os
from IPython.display import Image, display

roots = [Path(os.environ["QCC_DATASET_DIR"])] if "QCC_DATASET_DIR" in os.environ else [
    Path("/kaggle/input/qcc-local-sota-20261031"),
    Path("/kaggle/input/datasets/yoyobar/qcc-local-sota-20261031"),
]
DATASET = next((p for p in roots if (p / "portfolio-manifest.json").is_file()), None)
if DATASET is None:
    raise FileNotFoundError("Attach the public yoyobar/qcc-local-sota-20261031 dataset.")
display(Image(filename=str(DATASET / "submission-meme.png"), width=680))


## Short timeline

- **Invsqrt:** mrmeows / team selenium → busyaprime 63Q → yoyobar and GPT-6 Astra's 62Q circuit → our cone rewrites → Ahmed Berat Özer's phase correction → Ehan Mallick's two AND merges. The selected public 62Q circuit has a historical full64 PASS.
- **DQI:** public decoder and SomeJoint's 892Q work → Busya PRIME's product-caching improvements → our one-CCX reuse in this portfolio. Our earlier F3 888Q research removed 56 static CCX on a different circuit.
- **Chemistry:** Lera's factorization, public loaders and Heliosli's weights; we retain the latest selected public weights. Our older gauge-search work remains historical and is not claimed as applied to these weights.
- **Competition:** earlier Notebook v8 scored 23.17116, v10 scored 23.26622, and v11 scored 23.83433. These were different portfolios.


## Sources, credit and license

This work builds on public contributions. See the dataset's `CREDITS.md` for further lineage.

- **Invsqrt:** [Ehan Mallick](https://www.kaggle.com/code/ehanmallick/invsqrt-20-65-two-3-to-2-and-merges-on-y6225), busyaprime, mrmeows / team selenium, Ahmed Berat Özer and [manwithacat](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo); yoyobar and GPT-6 Astra contributed the 62Q and cone work.
- **DQI:** [Busya PRIME](https://www.kaggle.com/code/busyaprime/23-910-local-gm-exact-dqi-rewrites-892-qubits), [SomeJoint](https://www.kaggle.com/code/somejoint/quantum-circuits-23-85-local-gm-892-qubit-dqi), manwithacat, Heliosli, Ali Haydar Özdağ, GPT-6 Astra Pro, Claude and earlier public decoder contributors. Our added change is one direct product reuse.
- **Chemistry:** [Heliosli](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy), [Lera](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian), Dmitrii Gluzdov and upstream contributors.
- **Organizers:** Google Quantum AI / Qualtran for the task, verifier and SDK.

### License
Original notebook code and prose by yoyobar are licensed under the [Apache License 2.0](https://www.apache.org/licenses/LICENSE-2.0); a copy is included in the attached dataset. Third-party circuits, upstream materials and the supplied image retain their own terms. The DQI source lineage includes an earlier repository without an explicit license; this notebook's license does not relicense it.


## Submission file

The next cell packages the 18 pinned dataset files and checks each member's SHA-256 before and after writing `/kaggle/working/submission.zip`. It does not submit the entry or rerun scientific verification.


In [ ]:
# Copyright 2026 yoyobar
# SPDX-License-Identifier: Apache-2.0
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED, ZipInfo
import hashlib, json, os, shutil

manifest_path = DATASET / "portfolio-manifest.json"
with manifest_path.open("rb") as f:
    if hashlib.file_digest(f, "sha256").hexdigest() != "1bf3f97877d5811622fa2c57399e7e25530be2e6cbdada9e21bfced35dc21494":
        raise ValueError("Unexpected portfolio manifest identity")
manifest = json.loads(manifest_path.read_text())
expected = {name: row["sha256"] for name, row in manifest["entries"].items()}
names = {"invsqrt.kmb", "dqi.kmb", "dfthcsos/submission.npz"} | {f"dfthcsos/angle_k_{i}.kmb" for i in range(15)}
if set(expected) != names:
    raise ValueError("Unexpected manifest member set")

source = DATASET / "submission"
working = Path(os.environ.get("QCC_WORKING_DIR", "/kaggle/working"))
working.mkdir(parents=True, exist_ok=True)
temporary = working / "submission.tmp"
archive_path = working / "submission.zip"
try:
    with ZipFile(temporary, "w", compression=ZIP_DEFLATED, compresslevel=1) as archive:
        for name in sorted(names):
            path = source / name
            info = ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
            info.compress_type = ZIP_DEFLATED
            info.create_system = 3
            info.external_attr = 0o100644 << 16
            info.file_size = path.stat().st_size
            digest = hashlib.sha256()
            with path.open("rb") as input_file, archive.open(info, "w") as output_file:
                for chunk in iter(lambda: input_file.read(8 * 1024 * 1024), b""):
                    digest.update(chunk)
                    output_file.write(chunk)
            if digest.hexdigest() != expected[name]:
                raise ValueError(f"Unexpected circuit identity: {name}")
    with ZipFile(temporary) as z:
        if len(z.namelist()) != 18 or set(z.namelist()) != names:
            raise ValueError("Unexpected submission ZIP members")
        for name in sorted(names):
            with z.open(name) as f:
                if hashlib.file_digest(f, "sha256").hexdigest() != expected[name]:
                    raise ValueError(f"Packaged identity changed: {name}")
    temporary.replace(archive_path)
except Exception:
    temporary.unlink(missing_ok=True)
    raise
with archive_path.open("rb") as f:
    output_sha = hashlib.file_digest(f, "sha256").hexdigest()
(working / "submission-provenance.json").write_text(json.dumps({"submission_sha256": output_sha, "files": expected}, indent=2))
print("Ready:", archive_path)
print("18 circuit identities verified. ZIP SHA-256:", output_sha)
print("Packaging only; no verifier or competition submission was run by this cell.")
